In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import requests
import os
import re
from collections import Counter
from tqdm import tqdm

In [ ]:
def get_best_device():
    """
    Select the best available device for training: CUDA (NVIDIA GPU), MPS (Apple Silicon GPU), or CPU.
    Returns:
        torch.device: The selected device.
    """
    if torch.cuda.is_available():
        return torch.device('cuda')
    elif torch.backends.mps.is_available():
        return torch.device('mps')
    else:
        return torch.device('cpu')

def download_dataset(url, data_path):
    """
    Download the dataset from a URL if not already present locally.
    Args:
        url (str): The URL to download the dataset from.
        data_path (str): The local file path to save the dataset.
    Returns:
        str: The loaded text data.
    """
    if not os.path.exists(data_path):
        with open(data_path, 'w') as f:
            f.write(requests.get(url).text)
    with open(data_path, 'r') as f:
        text = f.read()
    return text

def tokenize(text):
    """
    Tokenize the input text into a list of words. For simplicity, this function
    splits on whitespace, removes punctuation, and lowercases the text.
    Args:
        text (str): The input text.
    Returns:
        list: List of words (tokens).
    """
    text = re.sub(r'[\W_]+', ' ', text.lower())
    return text.split()

def build_vocab(words):
    """
    Build vocabulary and mapping dictionaries from a list of words.
    Args:
        words (list): List of words in the dataset.
    Returns:
        tuple: (vocab, word2int, int2word)
    """
    vocab = sorted(set(words))
    word2int = {w: i for i, w in enumerate(vocab)}
    int2word = {i: w for i, w in enumerate(vocab)}
    return vocab, word2int, int2word

def encode(s, word2int):
    """
    Encode a string into a list of word indices using the vocabulary mapping.
    Args:
        s (str): Input string.
        word2int (dict): Mapping from word to index.
    Returns:
        list: List of word indices.
    """
    return [word2int[w] for w in tokenize(s)]

def decode(l, int2word):
    """
    Decode a list of word indices back into a string.
    Args:
        l (list): List of word indices.
        int2word (dict): Mapping from index to word.
    Returns:
        str: Decoded string.
    """
    return ' '.join([int2word[i] for i in l])

def get_batch(data, batch_size, block_size, device):
    """
    Generate a batch of input and target sequences for training.
    Args:
        data (Tensor): The full dataset as a tensor of word indices.
        batch_size (int): Number of sequences per batch.
        block_size (int): Length of each sequence (context window).
        device (torch.device): Device to move the batch to.
    Returns:
        tuple: (x, y) input and target tensors.
    """
    ix = torch.randint(len(data) - block_size - 1, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)


In [ ]:
class WordLSTM(nn.Module):
    """
    Multi-layer LSTM model for word-level language modeling.
    Args:
        vocab_size (int): Size of the vocabulary.
        hidden_size (int): Number of hidden units in each LSTM layer.
        num_layers (int): Number of LSTM layers.
        word2int (dict): Word to index mapping (for reference).
        int2word (dict): Index to word mapping (for reference).
    """
    def __init__(self, vocab_size, hidden_size, num_layers, word2int, int2word):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, hidden_size)
        self.lstm = nn.LSTM(hidden_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)
        self.word2int = word2int
        self.int2word = int2word

    def forward(self, x, hidden=None):
        """
        Forward pass through the model.
        Args:
            x (Tensor): Input tensor of word indices.
            hidden (tuple, optional): Hidden state for LSTM.
        Returns:
            tuple: (output logits, new hidden state)
        """
        x = self.embed(x)
        out, hidden = self.lstm(x, hidden)
        out = self.fc(out)
        return out, hidden

def train_model(model, data, vocab_size, device, epochs=2, batch_size=64, block_size=30, lr=2e-3, steps_per_epoch=200):
    """
    Train the LSTM model on the dataset.
    Args:
        model (nn.Module): The LSTM model.
        data (Tensor): The dataset as a tensor of word indices.
        vocab_size (int): Size of the vocabulary.
        device (torch.device): Device to train on.
        epochs (int): Number of epochs.
        batch_size (int): Batch size.
        block_size (int): Sequence length.
        lr (float): Learning rate.
        steps_per_epoch (int): Number of steps per epoch.
    Returns:
        nn.Module: The trained model.
    """
    optimizer = optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        pbar = tqdm(range(steps_per_epoch), desc=f"Epoch {epoch+1}")
        for step in pbar:
            # Get a batch of input word indices(x) and 
            # target word indices (y) sequences
            x, y = get_batch(data, batch_size, block_size, device)

            optimizer.zero_grad()
            # Forward pass: logits are the raw, unnormalized scores for
            # each word in the vocab
            # logits: (batch_size, block_size, vocab_size)
            logits, _ = model(x)

            # Reshape logits and targets for loss computation
            logits_flat = logits.view(-1, vocab_size)  # (batch_size*block_size, vocab_size)
            y_flat = y.view(-1)  # (batch_size*block_size)

            # Compute cross-entropy loss between predicted logits and true targets
            # loss: average negative log likelihood over the batch
            loss = loss_fn(logits_flat, y_flat)

            # Backpropagation and optimization step
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            if (step+1) % 50 == 0:
                pbar.set_postfix({'loss': loss.item()})

        print(f"Epoch {epoch+1} avg loss: {total_loss/steps_per_epoch:.4f}")
    return model

def train_lstm(device=None, block_size=32, batch_size=64, hidden_size=512, num_layers=3, epochs=3, lr=2e-3, steps_per_epoch=200):
    """
    Prepare data, build the model, and train the LSTM.
    Args:
        device (torch.device, optional): Device to use. If None, auto-selects best device.
    Returns:
        nn.Module: The trained LSTM model.
    """

    # Dataset
    url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
    data_path = 'tinyshakespeare.txt'
    text = download_dataset(url, data_path)
    words = tokenize(text)
    vocab, word2int, int2word = build_vocab(words)
    vocab_size = len(vocab)
    data = torch.tensor([word2int[w] for w in words], dtype=torch.long)

    # Device selection (CUDA > MPS > CPU)
    if device is None:
        device = get_best_device()
    print(f"Using device: {device}")

    # Model initialization
    model = WordLSTM(vocab_size, hidden_size, num_layers, word2int, int2word).to(device)

    # Train the model
    trained_model = train_model(model, data, vocab_size, device, epochs, batch_size, block_size, lr, steps_per_epoch)
    return trained_model

def generate_text(model, prompt, device=None, max_tokens=64):
    """
    Generate text from a trained model given a prompt.
    Args:
        model (nn.Module): The trained LSTM model.
        prompt (str): The initial text prompt.
        device (torch.device, optional): Device to use. If None, auto-selects best device.
        length (int): Number of words to generate.
    Returns:
        None. Prints the generated text.
    """
    # Device selection (CUDA > MPS > CPU)
    if device is None:
        device = get_best_device()
    print(f"Using device: {device}")

    model.eval()

    # Encode the prompt into word indices
    # input_ids: list of indices for the prompt words
    input_ids = encode(prompt, model.word2int)
    
    # Convert to tensor and add batch dimension
    # input_seq: shape (1, prompt_length)
    input_seq = torch.tensor(input_ids, dtype=torch.long).unsqueeze(0).to(device)
    
    hidden = None   # Initial hidden state (None = zeros)
    generated = list(input_ids)  # Start with the prompt

    for _ in range(max_tokens):
        # Forward pass: get logits for the next word
        # logits: (1, seq_len, vocab_size), only last position is relevant
        logits, hidden = model(input_seq, hidden)
        
        # Get probabilities for the next word (softmax over vocab)
        probs = torch.softmax(logits[:, -1, :], dim=-1)  # (1, vocab_size)

        # Sample the next word index from the probability distribution
        next_id = torch.multinomial(probs, num_samples=1).item()
        generated.append(next_id)

        # Prepare input for next step: the newly generated word
        input_seq = torch.tensor([[next_id]], dtype=torch.long).to(device)

    # Decode the generated indices back to words and print
    print("\nGenerated text:\n", decode(generated, model.int2word))


In [ ]:
# Train the LSTM model
model = train_lstm()

In [ ]:
# Generate text using the LSTM model
generate_text(model, prompt="O Romeo, Romeo, ", max_tokens=64)